# Chapter 37: ORB SLAM Architecture

<a href="../lite/lab/index.html?path=ch37_orbslam_architecture.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.distance import cdist

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

ORB SLAM is the Swiss Army knife of visual SLAM. Three threads running in
parallel: one tracks the camera in real time, one builds the map in the
background, and one watches for loop closures. Understanding its architecture
means understanding how all the pieces from the last 10 chapters fit together
into a working system.

This chapter builds each component from scratch:
- **Tracking thread:** match map to frame, estimate pose
- **Mapping thread:** select keyframes, triangulate new points
- **Loop closing thread:** detect revisits, correct drift
- **Place recognition:** bag of visual words for frame matching

```{admonition} What you will build
:class: tip

- Simulate the three parallel threads of ORB-SLAM: tracking, mapping, loop closing
- Implement keyframe selection based on feature displacement and tracking loss
- Build a simple bag of visual words for place recognition
- See how all the pieces from chapters 31 to 36 fit together into a working system

**Real world application:** ORB-SLAM is the most cited visual SLAM system. Understanding its architecture means understanding how to build production visual SLAM. After this chapter, you will see the full picture.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **ORB-SLAM3** | The reference implementation. Supports mono, stereo, RGBD, and IMU |
| **stella_vslam** | Community maintained fork of OpenVSLAM, compatible with ROS 2 |
| **RTAB-Map** | Alternative architecture with similar capabilities |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

### Shared Utilities

Before diving into each thread, we define shared camera projection and
rotation utilities used throughout the chapter.

In [ ]:
def rot_z(theta):
    """2D rotation matrix (rotation about z-axis)."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def project_2d(landmarks, pose):
    """Project 2D landmarks into a camera 'frame'.
    pose = (x, y, theta). Returns visibility mask and bearing angles."""
    x, y, theta = pose
    R = rot_z(-theta)
    t = np.array([x, y])
    local = (R @ (landmarks - t).T).T
    visible = local[:, 0] > 0.5
    angles = np.arctan2(local[:, 1], local[:, 0])
    fov_mask = np.abs(angles) < np.pi / 3  # 120 degree FOV
    mask = visible & fov_mask
    return mask, angles

def generate_circular_trajectory(n_frames, radius=5.0):
    """Generate poses on a circular trajectory."""
    angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
    poses = []
    for a in angles:
        x = radius * np.cos(a)
        y = radius * np.sin(a)
        theta = a + np.pi / 2
        poses.append((x, y, theta))
    return poses

## 37.1 Tracking Thread

The **tracking thread** runs on every frame. Its job:
1. Project known 3D map points into the current frame
2. Match with observed features
3. Estimate the camera pose using least squares (PnP style)

This must run in real time (~30 fps), so it only uses the existing map.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_map_pts = 30                   # 3D map points
obs_noise = 0.02                 # observation noise (radians)
pose_init_noise = 0.3            # initial pose guess noise
n_iterations = 20                # Gauss-Newton iterations
# ─────────────────────────────────────────────────────────────────────────────

landmarks = np.random.uniform(-8, 8, (n_map_pts, 2))
landmarks = landmarks[np.linalg.norm(landmarks, axis=1) > 2]
n_map_pts = len(landmarks)

true_pose = (1.0, 0.5, 0.3)
vis_mask, true_obs = project_2d(landmarks, true_pose)
noisy_obs = true_obs + np.random.normal(0, obs_noise, len(true_obs))
vis_idx = np.where(vis_mask)[0]

print(f'Map has {n_map_pts} landmarks')
print(f'Camera sees {len(vis_idx)} landmarks from pose '
      f'({true_pose[0]:.1f}, {true_pose[1]:.1f}, {true_pose[2]:.2f})')

In [ ]:
def tracking_cost(pose, landmarks_vis, obs_vis):
    """Compute residuals and Jacobian for pose estimation."""
    x, y, theta = pose
    R = rot_z(-theta)
    t = np.array([x, y])
    residuals = []
    J_rows = []
    for lm, obs in zip(landmarks_vis, obs_vis):
        local = R @ (lm - t)
        lx, ly = local
        predicted = np.arctan2(ly, lx)
        r = predicted - obs
        r = (r + np.pi) % (2 * np.pi) - np.pi
        residuals.append(r)
        denom = lx**2 + ly**2
        dangle_dlx = -ly / denom
        dangle_dly = lx / denom
        c, s = np.cos(theta), np.sin(theta)
        dx_lm = lm - t
        dlx_dx = -c;  dly_dx = s
        dlx_dy = s;   dly_dy = -c
        dlx_dth = s * dx_lm[0] + c * dx_lm[1]
        dly_dth = -c * dx_lm[0] + s * dx_lm[1]
        J_x = dangle_dlx * dlx_dx + dangle_dly * dly_dx
        J_y = dangle_dlx * dlx_dy + dangle_dly * dly_dy
        J_th = dangle_dlx * dlx_dth + dangle_dly * dly_dth
        J_rows.append([J_x, J_y, J_th])
    return np.array(residuals), np.array(J_rows)

pose_est = np.array([true_pose[0] + np.random.normal(0, pose_init_noise),
                     true_pose[1] + np.random.normal(0, pose_init_noise),
                     true_pose[2] + np.random.normal(0, pose_init_noise * 0.5)])

landmarks_vis = landmarks[vis_idx]
obs_vis = noisy_obs[vis_idx]
costs = []
poses_history = [pose_est.copy()]

for it in range(n_iterations):
    r, J = tracking_cost(pose_est, landmarks_vis, obs_vis)
    cost = 0.5 * np.sum(r**2)
    costs.append(cost)
    JtJ = J.T @ J
    Jtr = J.T @ r
    dx = np.linalg.solve(JtJ + 1e-6 * np.eye(3), -Jtr)
    pose_est += dx
    poses_history.append(pose_est.copy())

print(f'\nTracking results after {n_iterations} iterations:')
print(f'  True pose:      ({true_pose[0]:.3f}, {true_pose[1]:.3f}, {true_pose[2]:.3f})')
print(f'  Estimated pose: ({pose_est[0]:.3f}, {pose_est[1]:.3f}, {pose_est[2]:.3f})')
print(f'  Position error: {np.linalg.norm(pose_est[:2] - np.array(true_pose[:2])):.4f}')
print(f'  Angle error:    {abs(pose_est[2] - true_pose[2]):.4f} rad')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(landmarks[:, 0], landmarks[:, 1], c='steelblue', s=40, label='Map points')
ax.scatter(landmarks_vis[:, 0], landmarks_vis[:, 1], c='forestgreen', s=60,
           edgecolors='k', linewidth=0.5, label='Visible', zorder=5)
ph = np.array(poses_history)
ax.plot(ph[:, 0], ph[:, 1], 'orange', linewidth=1.5, marker='.', markersize=4, label='GN trajectory')
ax.scatter(*true_pose[:2], c='tomato', s=150, marker='*', zorder=10, label='True pose')
ax.scatter(pose_est[0], pose_est[1], c='forestgreen', s=150, marker='^', zorder=10, label='Final estimate')
for lm in landmarks_vis:
    ax.plot([pose_est[0], lm[0]], [pose_est[1], lm[1]], 'gray', alpha=0.2, linewidth=0.5)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Tracking: map projection + pose estimation', fontsize=13)
ax.legend(fontsize=9, loc='upper left'); ax.set_aspect('equal')

ax = axes[1]
ax.semilogy(costs, 'steelblue', linewidth=2, marker='o', markersize=4)
ax.set_xlabel('Iteration'); ax.set_ylabel('Cost (log scale)')
ax.set_title('Gauss Newton convergence', fontsize=13)

plt.tight_layout()
plt.show()

## 37.2 Mapping Thread

The **mapping thread** runs in the background. It decides when to add a new
**keyframe** and triangulates new map points between keyframes.

Not every frame becomes a keyframe. A keyframe is selected when:
- Enough time has passed since the last keyframe
- The camera has moved sufficiently
- The number of tracked features drops below a threshold

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_frames_map = 20
n_landmarks_map = 60
min_tracked_ratio = 0.6
min_frames_between_kf = 3
# ─────────────────────────────────────────────────────────────────────────────

lm_map = np.random.uniform(-10, 10, (n_landmarks_map, 2))
lm_map = lm_map[np.linalg.norm(lm_map, axis=1) > 3]
poses_map = [(i * 0.5 - 2, 0, 0.0) for i in range(n_frames_map)]

all_vis = []
for pose in poses_map:
    mask, _ = project_2d(lm_map, pose)
    all_vis.append(mask)

keyframes = [0]
kf_vis = all_vis[0]
tracked_counts = []
kf_map_sizes = [np.sum(all_vis[0])]

for i in range(1, n_frames_map):
    common = kf_vis & all_vis[i]
    tracked_ratio = np.sum(common) / max(np.sum(kf_vis), 1)
    tracked_counts.append(tracked_ratio)
    frames_since_kf = i - keyframes[-1]
    if tracked_ratio < min_tracked_ratio and frames_since_kf >= min_frames_between_kf:
        keyframes.append(i)
        kf_map_sizes.append(np.sum(all_vis[i]))
        kf_vis = all_vis[i]

print(f'Frames: {n_frames_map}')
print(f'Keyframes selected: {keyframes}')
print(f'Features visible per keyframe: {kf_map_sizes}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
pos_arr = np.array([(p[0], p[1]) for p in poses_map])
ax.plot(pos_arr[:, 0], pos_arr[:, 1], 'gray', linewidth=1, alpha=0.5)
ax.scatter(pos_arr[:, 0], pos_arr[:, 1], c='steelblue', s=20, label='Frames')
kf_pos = pos_arr[keyframes]
ax.scatter(kf_pos[:, 0], kf_pos[:, 1], c='tomato', s=100, marker='D',
           zorder=5, label=f'Keyframes ({len(keyframes)})', edgecolors='k')
ax.scatter(lm_map[:, 0], lm_map[:, 1], c='forestgreen', s=15, alpha=0.5, label='Landmarks')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Map: keyframe selection along trajectory', fontsize=13)
ax.legend(fontsize=9); ax.set_aspect('equal')

ax = axes[1]
ax.plot(range(1, n_frames_map), tracked_counts, 'steelblue', linewidth=2, marker='o', markersize=4)
ax.axhline(min_tracked_ratio, color='tomato', linewidth=2, linestyle='--',
           label=f'Threshold ({min_tracked_ratio})')
for kf in keyframes[1:]:
    ax.axvline(kf, color='orange', alpha=0.5, linewidth=1.5)
ax.set_xlabel('Frame'); ax.set_ylabel('Tracked feature ratio')
ax.set_title('Feature tracking ratio (orange = keyframe)', fontsize=13)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

## 37.3 Loop Closing Thread

The **loop closing** thread detects when the camera revisits a previously seen
place. When it does, it corrects the accumulated drift in the trajectory.

Detection uses a **similarity score** between the current keyframe's
descriptor and all previous keyframes.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_kf_loop = 20
loop_threshold = 0.7
# ─────────────────────────────────────────────────────────────────────────────

loop_poses = generate_circular_trajectory(n_kf_loop)
lm_loop = np.random.uniform(-10, 10, (100, 2))
lm_loop = lm_loop[np.linalg.norm(lm_loop, axis=1) > 3]

descriptors = []
for pose in loop_poses:
    mask, _ = project_2d(lm_loop, pose)
    noisy_mask = mask.astype(float) + np.random.normal(0, 0.1, len(mask))
    noisy_mask = np.clip(noisy_mask, 0, 1)
    descriptors.append(noisy_mask)
descriptors = np.array(descriptors)

norms = np.maximum(np.linalg.norm(descriptors, axis=1, keepdims=True), 1e-8)
desc_normed = descriptors / norms
similarity = desc_normed @ desc_normed.T

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.imshow(similarity, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine similarity')
ax.set_xlabel('Keyframe'); ax.set_ylabel('Keyframe')
ax.set_title('Keyframe similarity matrix', fontsize=13)

loop_detections = []
for i in range(n_kf_loop):
    for j in range(i + 5, n_kf_loop):
        if similarity[i, j] > loop_threshold:
            loop_detections.append((i, j, similarity[i, j]))
            ax.plot(j, i, 'ko', markersize=8)

ax = axes[1]
loop_pos = np.array([(p[0], p[1]) for p in loop_poses])
ax.plot(loop_pos[:, 0], loop_pos[:, 1], 'steelblue', linewidth=1.5)
ax.scatter(loop_pos[:, 0], loop_pos[:, 1], c='steelblue', s=30, zorder=5)
for i, j, sim in loop_detections:
    ax.plot([loop_pos[i, 0], loop_pos[j, 0]],
            [loop_pos[i, 1], loop_pos[j, 1]], 'tomato', linewidth=2, alpha=0.7)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Loop closures detected (red links)', fontsize=13)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print(f'Loop closures detected: {len(loop_detections)}')
for i, j, sim in loop_detections:
    print(f'  KF {i} <-> KF {j}: similarity = {sim:.3f}')

## 37.4 Keyframe Selection Policy

Good keyframe selection is critical. Too many keyframes waste computation.
Too few keyframes lose tracking. ORB SLAM uses multiple criteria:

1. **Minimum time** since last keyframe
2. **Median feature displacement** exceeds a threshold
3. **Number of tracked features** drops below a threshold

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_frames_kf = 50
n_features_total = 80
min_frame_gap = 3
displacement_thresh = 15.0
min_features_thresh = 40
# ─────────────────────────────────────────────────────────────────────────────

feature_base = np.random.uniform(50, 590, (n_features_total, 2))
speed = np.array([3.0, 0.5])
features_per_frame = []
alive = np.ones(n_features_total, dtype=bool)
positions = feature_base.copy()

for f in range(n_frames_kf):
    positions = positions + speed + np.random.normal(0, 1, positions.shape)
    out_of_frame = (positions[:, 0] < 0) | (positions[:, 0] > 640) | \
                   (positions[:, 1] < 0) | (positions[:, 1] > 480)
    random_loss = np.random.random(n_features_total) < 0.03
    alive = alive & ~out_of_frame & ~random_loss
    features_per_frame.append((positions.copy(), alive.copy()))

n_tracked = [np.sum(features_per_frame[i][1]) for i in range(n_frames_kf)]
kf_selected = [0]
kf_positions = features_per_frame[0][0].copy()
kf_alive = features_per_frame[0][1].copy()
displacements = [0]

for i in range(1, n_frames_kf):
    curr_pos, curr_alive = features_per_frame[i]
    common = kf_alive & curr_alive
    if np.sum(common) > 0:
        disp = np.median(np.linalg.norm(curr_pos[common] - kf_positions[common], axis=1))
    else:
        disp = float('inf')
    displacements.append(disp)
    frames_since = i - kf_selected[-1]
    if frames_since >= min_frame_gap:
        if disp > displacement_thresh or n_tracked[i] < min_features_thresh:
            kf_selected.append(i)
            kf_positions = curr_pos.copy()
            kf_alive = curr_alive.copy()

print(f'Total frames: {n_frames_kf}')
print(f'Keyframes selected: {kf_selected}')
print(f'Keyframe rate: {len(kf_selected)/n_frames_kf*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax = axes[0]
ax.plot(range(n_frames_kf), n_tracked, 'steelblue', linewidth=2, label='Tracked features')
ax.axhline(min_features_thresh, color='tomato', linewidth=2, linestyle='--',
           label=f'Threshold ({min_features_thresh})')
for kf in kf_selected:
    ax.axvline(kf, color='orange', alpha=0.4, linewidth=2)
ax.set_ylabel('Number of tracked features', fontsize=12)
ax.set_title('Keyframe selection: features tracked + displacement', fontsize=13)
ax.legend(fontsize=10)

ax = axes[1]
ax.plot(range(n_frames_kf), displacements, 'forestgreen', linewidth=2, label='Median displacement')
ax.axhline(displacement_thresh, color='tomato', linewidth=2, linestyle='--',
           label=f'Threshold ({displacement_thresh}px)')
for kf in kf_selected:
    ax.axvline(kf, color='orange', alpha=0.4, linewidth=2)
ax.set_xlabel('Frame'); ax.set_ylabel('Displacement (px)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 37.5 Place Recognition (Bag of Visual Words)

**Place recognition** identifies when the camera revisits a known place.
ORB SLAM uses a **bag of visual words** (BoVW) approach:

1. Each feature is assigned to a **visual word** (cluster center)
2. Each frame's signature is a **histogram** of visual word frequencies
3. Frames are matched by comparing histograms

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
vocab_size = 20
n_kf_bovw = 15
features_per_kf = 40
descriptor_dim_bovw = 8
# ─────────────────────────────────────────────────────────────────────────────

vocabulary = np.random.randn(vocab_size, descriptor_dim_bovw)

def make_descriptors(place_id, n_features, vocab, noise=0.5):
    rng = np.random.RandomState(place_id * 100)
    dominant_words = rng.choice(len(vocab), size=n_features // 2, replace=True)
    other_words = np.random.choice(len(vocab), size=n_features - len(dominant_words), replace=True)
    word_ids = np.concatenate([dominant_words, other_words])
    descs = vocab[word_ids] + np.random.randn(n_features, vocab.shape[1]) * noise
    return descs

place_ids = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

histograms = []
for kf_id in range(n_kf_bovw):
    descs = make_descriptors(place_ids[kf_id], features_per_kf, vocabulary)
    dists = cdist(descs, vocabulary)
    assignments = np.argmin(dists, axis=1)
    hist = np.bincount(assignments, minlength=vocab_size).astype(float)
    hist /= np.sum(hist)
    histograms.append(hist)
histograms = np.array(histograms)

In [ ]:
def histogram_intersection(h1, h2):
    return np.sum(np.minimum(h1, h2))

confusion = np.zeros((n_kf_bovw, n_kf_bovw))
for i in range(n_kf_bovw):
    for j in range(n_kf_bovw):
        confusion[i, j] = histogram_intersection(histograms[i], histograms[j])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
x_pos = np.arange(vocab_size)
ax.bar(x_pos - 0.2, histograms[0], 0.4, color='steelblue', alpha=0.7, label='KF 0 (place A)')
ax.bar(x_pos + 0.2, histograms[6], 0.4, color='tomato', alpha=0.7, label='KF 6 (place B)')
ax.set_xlabel('Visual word'); ax.set_ylabel('Frequency')
ax.set_title('BoVW histograms', fontsize=12); ax.legend(fontsize=9)

ax = axes[1]
ax.bar(x_pos - 0.2, histograms[0], 0.4, color='steelblue', alpha=0.7, label='KF 0 (place A)')
ax.bar(x_pos + 0.2, histograms[12], 0.4, color='forestgreen', alpha=0.7, label='KF 12 (place A again)')
ax.set_xlabel('Visual word')
ax.set_title('Same place, different time', fontsize=12); ax.legend(fontsize=9)

ax = axes[2]
im = ax.imshow(confusion, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Histogram intersection')
ax.set_xlabel('Keyframe'); ax.set_ylabel('Keyframe')
ax.set_title('Place recognition confusion matrix', fontsize=12)
ax.axhline(4.5, color='white', linewidth=1); ax.axhline(9.5, color='white', linewidth=1)
ax.axvline(4.5, color='white', linewidth=1); ax.axvline(9.5, color='white', linewidth=1)

plt.tight_layout()
plt.show()

print('Place recognition results:')
print(f'  KF0 vs KF6  (diff place): similarity = {confusion[0, 6]:.3f}')
print(f'  KF0 vs KF12 (same place): similarity = {confusion[0, 12]:.3f}')
print(f'  KF2 vs KF11 (same place): similarity = {confusion[2, 11]:.3f}')

## Capstone: Mini ORB SLAM Pipeline

We now combine all three threads into a simplified ORB SLAM pipeline:
1. **Tracking:** match map to frame, estimate pose
2. **Keyframe selection:** every 5th frame
3. **Mapping:** add new landmarks visible from new keyframes
4. **Loop closure:** detect revisit, correct trajectory

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_total_frames = 30
n_world_landmarks = 50
kf_interval = 5
odom_noise = 0.15
traj_radius = 5.0
# ─────────────────────────────────────────────────────────────────────────────

world_lm = np.random.uniform(-10, 10, (n_world_landmarks, 2))
world_lm = world_lm[np.linalg.norm(world_lm, axis=1) > 3.0]
n_world_landmarks = len(world_lm)

true_poses = generate_circular_trajectory(n_total_frames, traj_radius)
odom_poses = [true_poses[0]]
for i in range(1, n_total_frames):
    prev = odom_poses[-1]
    dx = true_poses[i][0] - true_poses[i-1][0] + np.random.normal(0, odom_noise)
    dy = true_poses[i][1] - true_poses[i-1][1] + np.random.normal(0, odom_noise)
    dth = true_poses[i][2] - true_poses[i-1][2] + np.random.normal(0, odom_noise * 0.3)
    odom_poses.append((prev[0] + dx, prev[1] + dy, prev[2] + dth))

true_xy = np.array([(p[0], p[1]) for p in true_poses])
odom_xy = np.array([(p[0], p[1]) for p in odom_poses])

map_points_cap = []; keyframe_ids_cap = []; n_new_per_kf_cap = []
seen_lm_ids_cap = set()
for frame_id in range(n_total_frames):
    vis_mask_f, _ = project_2d(world_lm, true_poses[frame_id])
    if frame_id % kf_interval == 0:
        keyframe_ids_cap.append(frame_id)
        new_count = 0
        for lm_id in np.where(vis_mask_f)[0]:
            if lm_id not in seen_lm_ids_cap:
                map_points_cap.append(world_lm[lm_id]); seen_lm_ids_cap.add(lm_id); new_count += 1
        n_new_per_kf_cap.append(new_count)
map_points_cap = np.array(map_points_cap)

print(f'Frames: {n_total_frames}, Keyframes: {keyframe_ids_cap}')
print(f'Map points: {len(map_points_cap)}, New per KF: {n_new_per_kf_cap}')

In [ ]:
corrected_xy = odom_xy.copy()
closure_error = odom_xy[-1] - odom_xy[0]
for i in range(n_total_frames):
    alpha = i / (n_total_frames - 1)
    corrected_xy[i] = odom_xy[i] - alpha * closure_error

odom_error = np.linalg.norm(odom_xy - true_xy, axis=1)
corrected_error = np.linalg.norm(corrected_xy - true_xy, axis=1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.plot(true_xy[:, 0], true_xy[:, 1], 'forestgreen', linewidth=2, label='True')
ax.plot(odom_xy[:, 0], odom_xy[:, 1], 'tomato', linewidth=2, linestyle='--', label='Odometry (drifted)')
ax.plot(corrected_xy[:, 0], corrected_xy[:, 1], 'steelblue', linewidth=2, linestyle=':', label='Loop corrected')
kf_xy_cap = true_xy[keyframe_ids_cap]
ax.scatter(kf_xy_cap[:, 0], kf_xy_cap[:, 1], c='orange', s=80, marker='D', zorder=5, edgecolors='k', label='Keyframes')
ax.scatter(map_points_cap[:, 0], map_points_cap[:, 1], c='gray', s=10, alpha=0.4, label='Map points')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Mini ORB SLAM: trajectory + map', fontsize=13)
ax.legend(fontsize=8, loc='lower left'); ax.set_aspect('equal')

ax = axes[1]
cumulative_pts = np.cumsum(n_new_per_kf_cap)
ax.bar(range(len(n_new_per_kf_cap)), n_new_per_kf_cap, color='steelblue', alpha=0.7)
ax2_twin = ax.twinx()
ax2_twin.plot(range(len(cumulative_pts)), cumulative_pts, 'tomato', linewidth=2, marker='o', markersize=5)
ax.set_xlabel('Keyframe index'); ax.set_ylabel('New map points', color='steelblue')
ax2_twin.set_ylabel('Total map points', color='tomato')
ax.set_title('Map growth over keyframes', fontsize=13)

ax = axes[2]
ax.plot(range(n_total_frames), odom_error, 'tomato', linewidth=2, label='Before loop closure')
ax.plot(range(n_total_frames), corrected_error, 'steelblue', linewidth=2, label='After loop closure')
ax.set_xlabel('Frame'); ax.set_ylabel('Position error (m)')
ax.set_title('Loop closure reduces drift', fontsize=13); ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'Odometry RMSE:       {np.sqrt(np.mean(odom_error**2)):.3f} m')
print(f'Loop corrected RMSE: {np.sqrt(np.mean(corrected_error**2)):.3f} m')

**Capstone takeaways:**
- The tracking thread estimates pose from the existing map at every frame
- The mapping thread adds keyframes and new map points incrementally
- The loop closing thread detects revisits and corrects drift
- Even a simple linear correction from loop closure dramatically reduces trajectory error
- In real ORB SLAM, pose graph optimization distributes the correction more intelligently

---

## Exercises

### Exercise 37.1
Modify the tracking thread to use **Huber robust loss** instead of squared loss.
Add 3 outlier observations with 1 radian of noise. Show that Huber loss is more
robust than squared loss by comparing the final pose estimate.

In [ ]:
# Your code here

### Exercise 37.2
Implement an **adaptive keyframe selection** strategy: reduce the minimum frame
gap when the camera moves fast and increase it when the camera is stationary.
Test on a trajectory with varying speed.

In [ ]:
# Your code here

### Exercise 37.3
Test the place recognition system with three distinct places (A, B, C) visited
in the pattern A, B, C, A, B. Plot the confusion matrix and verify that the
system correctly identifies all revisits.

In [ ]:
# Your code here

### Exercise 37.4
In the capstone, replace the linear loop closure correction with a **pose graph
optimization** approach. Define odometry edges and one loop closure edge, then
run Gauss Newton to optimize all poses simultaneously.

In [ ]:
# Your code here